# Weekday Ridership and Stop Data Cleaning, Standardization, and Integration Pipeline

This script prepares and integrates transit ridership data with stop-level data by cleaning inconsistencies, applying agency-specific fixes, and merging datasets using appropriate matching strategies.

**Summary**
- Load & filter data: Ridership data is standardized and filtered to weekdays; stop data is aligned with consistent agency names.
- Clean data: Fixes inconsistencies in route_id, stop_id, and stop_code across agencies to improve matching.
- Split by strategy: Agencies are grouped based on how they can be matched (stop_id, stop_code, or stop_name).
- Merge datasets: Each group is merged with stop data, keeping only successful matches.
- Combine results: All matched data is concatenated into a final dataset linking weekday ridership with stops and routes.

In [1]:
pip install shared_utils

ERROR: Could not find a version that satisfies the requirement shared_utils (from versions: none)
ERROR: No matching distribution found for shared_utils
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()
from rapidfuzz import process, fuzz

pd.set_option('display.max_columns', None)

In [3]:
# Read in ridership data 
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
ridership_data_weekday = (
    pd.read_csv(f"{GCS_FILE_PATH}/ridership_all.csv")
    .sort_values(by='stop_name')
    .loc[lambda df: df['day_type'] == "Weekday"]
    .reset_index(drop=True)
)

ridership_data_weekday.info()
ridership_data_weekday.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21887 entries, 0 to 21886
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   organization_name         21887 non-null  object 
 1   stop_id                   21812 non-null  object 
 2   stop_name                 18008 non-null  object 
 3   stop_lat                  6341 non-null   float64
 4   stop_lon                  6341 non-null   float64
 5   day_type                  21887 non-null  object 
 6   average_daily_boardings   21887 non-null  float64
 7   average_daily_alightings  19676 non-null  float64
 8   start_date                21887 non-null  object 
 9   end_date                  21887 non-null  object 
dtypes: float64(4), object(6)
memory usage: 1.7+ MB


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
0,Gold Coast Transit,VNACLR1,.,34.343645,-119.294028,Weekday,0.000000,3.000000,2025-05-01,2025-05-31
1,Samtrans,345017,1000 El Camino Real-Menlo College,37.457839,-122.190513,Weekday,9.523810,24.571429,2025-08-01,2025-08-31
2,Golden Gate Transit,40469,1011 Andersen Dr,NaN,NaN,Weekday,1.909091,0.000000,2025-09-01,2025-09-30
3,Samtrans,343119,1011 Crestview Dr,37.484237,-122.284042,Weekday,1.444444,1.888889,2025-08-01,2025-08-31
4,Samtrans,340606,1060 Carolan Ave,37.587028,-122.359191,Weekday,10.333333,5.500000,2025-08-01,2025-08-31


In [5]:
agency_mapping = {
    'Gold Coast Schedule': 'Gold Coast Transit',
    'Burbank Schedule': 'City of Burbank',
    'SamTrans Schedule': 'Samtrans',
    'Big Blue Bus Schedule': 'Big Blue Bus',
    'Foothill Schedule': 'Foothill Transit',
    'San Diego Schedule': 'SDMTS',
    'Golden Gate Park Shuttle Schedule': 'Golden Gate Park Shuttle',
    'Fresno Schedule': 'Fresno County',
    'OmniTrans Schedule': 'Omnitrans',
    'Sacramento Schedule': 'SacRT Bus',
    'Bay Area 511 BART Schedule': 'San Francisco Bay Area Rapid Transit District',
    'Riverside Schedule': 'Riverside Transit',
    'OCTA Schedule': 'Orange County Transportation Authority',
    'Santa Cruz Schedule': 'Santa Cruz Metro',
    'Long Beach Schedule': 'Long Beach Transit',
    'Caltrain Schedule': 'Caltrain',
    'SBMTD Schedule': 'SBMTD',
    'Culver City Schedule': 'Culver City Bus',
    'Bay Area 511 Golden Gate Transit Schedule': 'Golden Gate Transit'
}

### Weekday Stops Data Processing

In [6]:
stops_aggregated_weekday = (
    pd.read_csv(f"{GCS_FILE_PATH}/stops_aggregated_weekday.csv")
    .assign(organization_name=lambda df: df['name'].map(agency_mapping))
)

stops_aggregated_weekday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27355 entries, 0 to 27354
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   feed_key           27355 non-null  object
 1   stop_id            27355 non-null  object
 2   stop_name          27355 non-null  object
 3   n_arrivals         27355 non-null  int64 
 4   n_routes           27355 non-null  int64 
 5   route_id_list      27355 non-null  object
 6   stop_code          24376 non-null  object
 7   pt_geom            27355 non-null  object
 8   name               27355 non-null  object
 9   organization_name  27000 non-null  object
dtypes: int64(2), object(8)
memory usage: 2.1+ MB


In [7]:
# Upon Checking found that Samtrans Transit Route Id has -216 in the stops aggregated data, while ridership data we have doesnot. So removing -216 from stopsa_aggregated data 
# stops_aggregated_weekday.loc[
#     stops_aggregated_weekday['name'] == "SamTrans Schedule",
#     'route_id'
# ] = stops_aggregated_weekday.loc[
#     stops_aggregated_weekday['name'] == "SamTrans Schedule",
#     'route_id'
# ].str.replace('-216$', '', regex=True)


# Upon checking, found that Gold Coast Schedule stop_ids have some issues:
# 1. Some stop_ids contain ':' which prevents correct merge with ridership data
# 2. Two specific stop_ids ('GCTDMAIN1' and 'SAClMAI1') do not match the ridership data stop_ids ('GCTDMAIN' and 'SACLMAI1')
# So we remove colons and update these two stop_ids before merging
stops_aggregated_weekday.loc[
    stops_aggregated_weekday['name'] == "Gold Coast Schedule",
    'stop_id'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['name'] == "Gold Coast Schedule",
    'stop_id'
].str.replace(':', '', regex=False).replace({
    'GCTDMAIN1': 'GCTDMAIN',
    'SAClMAI1': 'SACLMAI1'
})



# Upon checking, found that Some stops are unmatched because Big Blue Bus ridership uses negative stop_codes.
# These correspond to valid stop_codes in the stop data (same route and stop_name).
# The mapping below replaces negative ridership stop_codes with the correct stop_codes
# so they can match during the merge.

# Apply only for Big Blue Bus
mask_bb = ridership_data_weekday['organization_name'] == 'Big Blue Bus'

# Convert to int first
ridership_data_weekday.loc[mask_bb, 'stop_id'] = (
    ridership_data_weekday.loc[mask_bb, 'stop_id']
    .astype(int)
    .replace({
        -4: 'MNSWSMNF',
        -5: '014MCHNN',
        -6: '014PCONF',
        -7: '014PCOSF',
        -8: '017PRLNN',
        -9: '017PRLSF',
        -13: 'PRL014EF',
        -14: 'PRL014WN',
        -18: 'SMCBUNPL'
    })
    .astype(str)
)


stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Big Blue Bus',
    'stop_code'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Big Blue Bus',
    'stop_code'
].astype(str).str.strip()


# Upon checking found that some route_ids are differently named in ridership data compared to stop_data so changing that to match
# Filter for Culver City Bus
# mask_cc = ridership_data_weekday['organization_name'] == 'Culver City Bus'

# # Standardize route_id in ridership
# ridership_data_weekday.loc[mask_cc, 'route_id'] = (
#     ridership_data_weekday.loc[mask_cc, 'route_id']
#     .replace({
#         '1C1': '1c1',
#         'Rapid 6': '6R'
#     })
#)

# # Standardize route_id in stops
# stops_aggregated_weekday.loc[
#     stops_aggregated_weekday['organization_name'] == 'Culver City Bus',
#     'route_id'
# ] = stops_aggregated_weekday.loc[
#     stops_aggregated_weekday['organization_name'] == 'Culver City Bus',
#     'route_id'
# ].replace({
#     '1C1': '1c1',
#     'Rapid 6': '6R'
# })


# Upon checking found that Long Beach transit and OCTA stop_id has leading zeros like 0110, 0220 and so on
for agency in ['Long Beach Transit', 'Orange County Transportation Authority']:
    stops_aggregated_weekday.loc[
        stops_aggregated_weekday['organization_name'] == agency,
        'stop_id'
    ] = (
        stops_aggregated_weekday.loc[
            stops_aggregated_weekday['organization_name'] == agency,
            'stop_id'
        ]
        .str.replace(r'^0+', '', regex=True)
    )




In [8]:
master_cols = [
    "organization_name",
    "feed_key",
    "stop_id",
    "stop_name",
    "stop_code",
    "n_arrivals",
    "n_routes",
    "pt_geom",
    "day_type",
    "route_id_list",
    "average_daily_boardings",
    "average_daily_alightings",
    "start_date",
    "end_date"
]

def standardize_columns(df, master_cols):
    
    # add missing columns
    for col in master_cols:
        if col not in df.columns:
            df[col] = None

    # keep only master columns and order them
    return df[master_cols]

### Agencies with Weekday Data, Categorized by Match Type: route*stop_id, stop_code, or stop_name

In [9]:
agencies__with_stop_id_match = [
    'Foothill Transit',
    'Gold Coast Transit',
    'Golden Gate Transit',
    'Long Beach Transit',
    'Orange County Transportation Authority',
    # 'Omnitrans',    
    'SacRT Bus',
    'Samtrans',
    'SDMTS',
    'San Francisco Bay Area Rapid Transit District', 
    'Fresno County',
    # 'SBMTD'
]

agencies_with_stop_code_match = [
    'Culver City Bus',
    'Big Blue Bus',
    'Riverside Transit',
    # 'Santa Cruz Metro',
    'Golden Gate Park Shuttle',
]



agencies_with_stop_name_match = [
                          'Caltrain',
                          'City of Burbank',
                         ]

In [10]:
# Split Ridership by categories

def split_ridership_by_agency(ridership_df, agency_groups):
    subsets = {}
    for name, agencies in agency_groups.items():
        subsets[name] = ridership_df[ridership_df['organization_name'].isin(agencies)]
    return subsets

agency_groups = {
    'with_stop_id_match': agencies__with_stop_id_match,
    'with_stop_code_match': agencies_with_stop_code_match,
    'with_stop_name_match': agencies_with_stop_name_match
}

ridership_subsets = split_ridership_by_agency(ridership_data_weekday, agency_groups)



In [11]:
# # Merge with route_id
# merge_with_route = (
#     pd.merge(
#         ridership_subsets['with_route'],  
#         stops_aggregated_weekday,
#         on=['organization_name', 'route_id', 'stop_id'],
#         how='left',
#         indicator=True,
#         suffixes=('_x','_y')
#     )
#     .rename(columns={'stop_name_y': 'stop_name'})  # rename matched stop_name
# )

# # Keep only rows where the merge matched in both DataFrames
# merge_with_route_matched = merge_with_route[merge_with_route['_merge'] == 'both'].copy()


# # Optional: inspect merge results
# merge_counts = merge_with_route['_merge'].value_counts()
# print(merge_counts)

In [12]:
# merge_with_route_matched = standardize_columns(merge_with_route_matched, master_cols)

In [13]:
# Merge without route_id
merge_with_stop_id_match = (
    pd.merge(
        ridership_subsets['with_stop_id_match'], 
        stops_aggregated_weekday,
        on=['organization_name', 'stop_id'],
        how='left',
        indicator=True,
        suffixes=('_x','_y')
    )
    .rename(columns={
        'stop_name_y': 'stop_name',
    })
)

# Keep only rows where the merge matched in both DataFrames
merge_with_stop_id_matched = merge_with_stop_id_match[merge_with_stop_id_match['_merge'] == 'both'].copy()

merge_counts = merge_with_stop_id_matched['_merge'].value_counts()
print(merge_counts)

_merge
both          17741
left_only         0
right_only        0
Name: count, dtype: int64


In [14]:
merge_with_stop_id_matched = standardize_columns(merge_with_stop_id_matched, master_cols)
merge_with_stop_id_matched.sample(5)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,route_id_list,average_daily_boardings,average_daily_alightings,start_date,end_date
3707,SDMTS,1fff52f9349da228c56eef492df5001b,13001,Camino De La Reina & Mission Valley West,13001,47.0,1.0,POINT(-117.15664603 32.76751308),Weekday,['6'],8.458650,15.227797,2024-09-01,2025-01-25
6311,Orange County Transportation Authority,cd299184726656597ae2cdb4f4e81e4a,4502,GOLDEN LANTERN-DUNES,4502,23.0,1.0,POINT(-117.681331 33.512979),Weekday,['90'],0.000000,0.000000,2025-02-04,2025-02-04
13041,Long Beach Transit,cddd375786d835389a7beb9632369907,2846,Rosecrans & Downey SE,2846,27.0,1.0,POINT(-118.151152 33.903632),Weekday,['71'],3.772599,120.655985,2024-07-01,2025-06-30
1898,Orange County Transportation Authority,cd299184726656597ae2cdb4f4e81e4a,4897,BAYSIDE-EAST PROMONTORY POINT,4897,41.0,1.0,POINT(-117.895938 33.61183),Weekday,['55'],6.000000,6.000000,2025-02-04,2025-02-04
13368,Fresno County,23d1893801eefadf7544a670a3bcd312,890,SE VENTURA - HAZELWOOD,890,58.0,1.0,POINT(-119.774731 36.735775),Weekday,['3990'],1.162876,2.022160,2024-09-01,2025-08-31


In [15]:
merge_with_stop_code_match = (
    pd.merge(
        ridership_subsets['with_stop_code_match'],  
        stops_aggregated_weekday,
        left_on=['organization_name', 'stop_id'],  # ridership stop_id
        right_on=['organization_name', 'stop_code'],  # stops table stop_code
        how='left',
        indicator=True
    )
    .rename(columns={
        'stop_id_y': 'stop_id',
        'stop_name_y': 'stop_name',
    })
)


# Keep only rows where the merge matched in both DataFrames
merge_with_stop_code_matched = merge_with_stop_code_match[merge_with_stop_code_match['_merge'] == 'both'].copy()

# Check merge results
merge_counts = merge_with_stop_code_matched['_merge'].value_counts()
print(merge_counts)

_merge
both          3449
left_only        0
right_only       0
Name: count, dtype: int64


In [16]:
merge_with_stop_code_matched = standardize_columns(merge_with_stop_code_matched, master_cols)
merge_with_stop_code_matched.sample(5)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,route_id_list,average_daily_boardings,average_daily_alightings,start_date,end_date
1486,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,293,Magnolia + Harrison,1165,98.0,4.0,POINT(-117.453449 33.916621),Weekday,"['1', '10', '12', '21']",83.930108,NaN,2025-01-01,2025-10-31
2397,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,1030,Alessandro + Century,2207,36.0,2.0,POINT(-117.351034 33.936869),Weekday,"['20', '22']",3.961326,NaN,2025-01-01,2025-10-31
1246,Big Blue Bus,7a3f513c343b16a30c135ed7d332b6d6,547,WILSHIRE BLVD & 20TH ST,1402,41.0,1.0,POINT(-118.483135 34.031603),Weekday,['2'],19.670595,20.552945,2024-08-01,2025-11-30
1509,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,1768,Main + Metro,1193,14.0,1.0,POINT(-117.564689 33.885645),Weekday,['3'],5.050562,NaN,2025-01-01,2025-10-31
897,Big Blue Bus,7a3f513c343b16a30c135ed7d332b6d6,796,PICO BLVD & ROXBURY DR,2371,77.0,1.0,POINT(-118.401974 34.054157),Weekday,['7'],27.138735,26.018952,2024-08-01,2025-11-30


In [17]:
cols_to_keep = [
    'feed_key', 'stop_id',
    'n_arrivals', 'n_routes', 'pt_geom', 'route_id_list'
]

def fuzzy_match_subset(left_df, right_df, threshold=80):
    results = []

    for org in left_df['organization_name'].unique():
        left_group = left_df[left_df['organization_name'] == org]
        right_group = right_df[right_df['organization_name'] == org]

        if right_group.empty:
            continue

        choices = right_group['stop_name'].tolist()

        for idx, row in left_group.iterrows():
            match, score, match_idx = process.extractOne(
                row['stop_name'],
                choices,
                scorer=fuzz.token_set_ratio
            )

            if score >= threshold:
                matched_row = right_group.iloc[match_idx]
                subset = matched_row[cols_to_keep].to_dict()
                matched_name = matched_row['stop_name']
            else:
                subset = {col: None for col in cols_to_keep}
                matched_name = None

            subset.update({
                'left_index': idx,
                'original_stop_name': row['stop_name'],
                'matched_stop_name': matched_name,
                'score': score
            })

            results.append(subset)

    return pd.DataFrame(results)

matches_df = fuzzy_match_subset(
    ridership_subsets['with_stop_name_match'], 
    stops_aggregated_weekday,
    threshold=80
)

In [18]:
merge_with_stop_name_match = (
    ridership_subsets['with_stop_name_match']
    .merge(
        matches_df,
        left_index=True,
        right_on='left_index',
        how='left',
        indicator=True
    )
    .rename(columns={
        'stop_id_y': 'stop_id',
    })
)

# Keep only rows where the merge matched in both DataFrames
merge_with_stop_name_matched = merge_with_stop_name_match[merge_with_stop_name_match['_merge'] == 'both'].copy()

merge_counts = merge_with_stop_name_matched['_merge'].value_counts()
print(merge_counts)

_merge
both          75
left_only      0
right_only     0
Name: count, dtype: int64


In [19]:
merge_with_stop_name_matched = standardize_columns(merge_with_stop_name_matched, master_cols)
merge_with_stop_name_matched.sample(5)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,route_id_list,average_daily_boardings,average_daily_alightings,start_date,end_date
28,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70221,Sunnyvale,None,52.0,3.0,POINT(-122.031372 37.378916),Weekday,"['77119', '77121', '77122']",1408.195136,NaN,2023-11-01,2025-07-31
45,City of Burbank,cc6a68a39d22c29b49116584971e69a8,3068904,Burbank & Whitehall,None,51.0,1.0,POINT(-118.356776 34.172425),Weekday,['3163'],0.681818,1.045455,2024-05-01,2024-05-31
48,City of Burbank,cc6a68a39d22c29b49116584971e69a8,3067456,Empire and Niagara,None,51.0,1.0,POINT(-118.342181 34.192016),Weekday,['3163'],0.454545,1.227273,2024-05-01,2024-05-31
5,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70081,Burlingame,None,37.0,1.0,POINT(-122.3449 37.580197),Weekday,['77119'],542.005575,NaN,2023-11-01,2025-07-31
23,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70261,San Jose Diridon,None,56.0,4.0,POINT(-121.903011 37.329239),Weekday,"['77119', '77121', '77122', '77123']",1789.561827,NaN,2023-11-01,2025-07-31


In [20]:
all_ridership_stop_trips_weekday_data = pd.concat([
    merge_with_stop_id_matched,
    merge_with_stop_code_matched,
    merge_with_stop_name_matched    
], ignore_index=True)

In [21]:
all_ridership_stop_trips_weekday_data.head(5)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,route_id_list,average_daily_boardings,average_daily_alightings,start_date,end_date
0,Gold Coast Transit,3cb676436aad669e52042c0e97a9a051,VNACLR1,.,NaN,14.0,1.0,POINT(-119.294028 34.343645),Weekday,['16'],0.000000,3.000000,2025-05-01,2025-05-31
1,Samtrans,db97cc02836aa5f0cf647d80160b23ec,345017,1000 El Camino Real-Menlo College,345017,72.0,1.0,POINT(-122.191284 37.457543),Weekday,['ECR-216'],9.523810,24.571429,2025-08-01,2025-08-31
2,Samtrans,db97cc02836aa5f0cf647d80160b23ec,343119,1011 Crestview Dr,343119,5.0,1.0,POINT(-122.284103 37.484282),Weekday,['61-216'],1.444444,1.888889,2025-08-01,2025-08-31
3,Samtrans,db97cc02836aa5f0cf647d80160b23ec,340606,1060 Carolan Ave,340606,7.0,1.0,POINT(-122.359627 37.586685),Weekday,['46-216'],10.333333,5.500000,2025-08-01,2025-08-31
4,SDMTS,1fff52f9349da228c56eef492df5001b,12049,10th Av & A St,12049,18.0,3.0,POINT(-117.15569071 32.71857729),Weekday,"['110', '280', '290']",3.485637,42.889568,2024-09-01,2025-01-25


In [22]:
all_ridership_stop_trips_weekday_data.to_csv(f"{GCS_FILE_PATH}/ridership_trips_routes_weekday.csv", index=False)